In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
customer_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("customer_name", StringType(), False),
    StructField("city", StringType(), True),
    StructField("balance", DoubleType(), True)
])

In [0]:
df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(customer_schema)
    .load("/Volumes/banking/bronze/landing_volume/customers.csv")
    )

In [0]:
display(df)

In [0]:
df.printSchema()

In [0]:
df.count()

In [0]:
from pyspark.sql.functions import col, when

df = df.withColumn(
    "customer_type",
    when(col("balance") >= 75000, "Premium")
    .otherwise("Regular")
)

In [0]:
display(df)

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("banking.bronze.customer_master")

In [0]:
df = df.withColumn(
    "balance",
    col("balance").cast("decimal(12,2)")
)

In [0]:
df.printSchema()

In [0]:
df.write \
  .mode("overwrite") \
  .format("delta") \
  .saveAsTable("banking.bronze.customer_master")

In [0]:
%sql
select * from banking.bronze.customer_master;